## Stock Price Prediction Prototype

This notebook walks through the three parts of the S&P 500 daily price data: storing the data efficiently, building technical indicators with two prediction models, and finishing with an interactive dashboard.

The file `all_stocks_5yr.csv` is expected in `data/csv/` inside the working directory, and everything the notebook writes goes to `data/parquet/`. The reasoning behind each technology choice (Parquet, Polars, Dash) is covered in the accompanying `readme` file, so here we simply use them.

In [ ]:
# Run once if a package is missing
%pip install -q polars pyarrow scikit-learn plotly dash

In [4]:
import os
import numpy as np
import polars as pl
import plotly.graph_objects as go
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor

CSV_FILE      = "data/csv/all_stocks_5yr.csv"      # input (read only)
PARQUET_DIR   = "data/parquet"                      # all output is written here
PARQUET_FILE  = f"{PARQUET_DIR}/all_stocks_5yr.parquet"
FEATURE_FILE  = f"{PARQUET_DIR}/stocks_features.parquet"
PRED_FILE     = f"{PARQUET_DIR}/predictions_test.parquet"
FORECAST_FILE = f"{PARQUET_DIR}/forecast_next_day.parquet"

os.makedirs(PARQUET_DIR, exist_ok=True)

## Part 1 – Storing and retrieving the data

We convert the CSV to a Parquet file with zstd compression. The conversion is done lazily, so Polars streams the file through instead of loading it all into memory. That is what lets the same code work when the data grows 10x or 100x.

In [7]:
(
    pl.scan_csv(CSV_FILE, try_parse_dates=True)
      .sink_parquet(PARQUET_FILE, compression="zstd", compression_level=3)
)
print("Parquet file written:", PARQUET_FILE)

Parquet file written: data/parquet/all_stocks_5yr.parquet


Quick sanity check on the conversion: file sizes side by side and the row count read straight from the Parquet metadata. It should match the 619,040 rows in the assignment brief.

In [10]:
sizes = pl.DataFrame({
    "file": [CSV_FILE, PARQUET_FILE],
    "size_mb": [round(os.path.getsize(CSV_FILE) / 1e6, 2),
                round(os.path.getsize(PARQUET_FILE) / 1e6, 2)],
})
n_rows = pl.scan_parquet(PARQUET_FILE).select(pl.len()).collect().item()
print(f"Rows in Parquet file: {n_rows:,}")
sizes

Rows in Parquet file: 619,040


file,size_mb
str,f64
"""data/csv/all_stocks_5yr.csv""",30.1
"""data/parquet/all_stocks_5yr.pa…",9.42


Now a retrieval example. Asking for one ticker and only four columns means Polars reads just the needed columns and skips row groups that cannot contain that ticker, instead of scanning everything.

In [13]:
aapl = (
    pl.scan_parquet(PARQUET_FILE)
      .filter(pl.col("name") == "AAPL")
      .select(["date", "open", "close", "volume"])
      .collect()
)
print(aapl.shape)
aapl.head()

(1259, 4)


date,open,close,volume
date,f64,f64,i64
2013-02-08,67.7142,67.8542,158168416
2013-02-11,68.0714,68.5614,129029425
2013-02-12,68.5014,66.8428,151829363
2013-02-13,66.7442,66.7156,118721995
2013-02-14,66.3599,66.6556,88809154


## Part 2 – Analysis, indicators and prediction models

We load the Parquet file and keep only rows with a closing price and volume, since those are what our indicators rely on. Sorting by ticker and date guarantees every calculation runs in time order within each company.

In [16]:
prices = (
    pl.scan_parquet(PARQUET_FILE)
      .filter(pl.col("close").is_not_null() & pl.col("volume").is_not_null())
      .sort(["name", "date"])
      .collect()
)
print(prices.shape)
display(prices.null_count())
prices.head()

(619040, 7)


date,open,high,low,close,volume,name
u32,u32,u32,u32,u32,u32,u32
0,11,8,8,0,0,0


date,open,high,low,close,volume,name
date,f64,f64,f64,f64,i64,str
2013-02-08,45.07,45.35,45.0,45.08,1824755,"""A"""
2013-02-11,45.17,45.18,44.45,44.6,2915405,"""A"""
2013-02-12,44.81,44.95,44.5,44.62,2373731,"""A"""
2013-02-13,44.81,45.24,44.68,44.75,2052338,"""A"""
2013-02-14,44.72,44.78,44.36,44.58,3826245,"""A"""


The null counts above show whether anything is missing in the columns we use. Next come the first indicators: a 20-day simple moving average (SMA), a 20-day exponential moving average (EMA), a few return windows, and today's volume relative to its 20-day average. Every calculation uses `.over("name")` so one company never bleeds into another.

In [19]:
prices = prices.with_columns(
    pl.col("close").rolling_mean(window_size=20).over("name").alias("sma_20"),
    pl.col("close").ewm_mean(span=20, adjust=False).over("name").alias("ema_20"),
    pl.col("close").pct_change().over("name").alias("ret_1"),
    pl.col("close").pct_change(5).over("name").alias("ret_5"),
    pl.col("close").pct_change(10).over("name").alias("ret_10"),
    (pl.col("volume") / pl.col("volume").rolling_mean(window_size=20).over("name")).alias("vol_ratio"),
)
prices.select(["name", "date", "close", "sma_20", "ema_20", "ret_1", "vol_ratio"]).tail(3)

name,date,close,sma_20,ema_20,ret_1,vol_ratio
str,date,f64,f64,f64,f64,f64
"""ZTS""",2018-02-05,73.83,76.6965,76.361598,-0.038421,1.021483
"""ZTS""",2018-02-06,73.27,76.648,76.06716,-0.007585,1.661167
"""ZTS""",2018-02-07,73.86,76.5855,75.856954,0.008052,1.484411


Third indicator: the 14-day Relative Strength Index (RSI), which measures how strong recent gains are compared with recent losses on a 0 to 100 scale. We use Wilder's smoothing and drop the helper columns once RSI is built.

In [22]:
prices = (
    prices
    .with_columns(pl.col("close").diff().over("name").alias("delta"))
    .with_columns(
        pl.when(pl.col("delta") > 0).then(pl.col("delta")).otherwise(0.0).alias("gain"),
        pl.when(pl.col("delta") < 0).then(-pl.col("delta")).otherwise(0.0).alias("loss"),
    )
    .with_columns(
        pl.col("gain").ewm_mean(alpha=1 / 14, adjust=False).over("name").alias("avg_gain"),
        pl.col("loss").ewm_mean(alpha=1 / 14, adjust=False).over("name").alias("avg_loss"),
    )
    .with_columns(
        (100 - 100 / (1 + pl.col("avg_gain") / pl.col("avg_loss"))).fill_nan(50.0).alias("rsi_14")
    )
    .drop(["delta", "gain", "loss", "avg_gain", "avg_loss"])
)
prices.select(["name", "date", "close", "rsi_14"]).tail(3)

name,date,close,rsi_14
str,date,f64,f64
"""ZTS""",2018-02-05,73.83,41.637053
"""ZTS""",2018-02-06,73.27,39.741833
"""ZTS""",2018-02-07,73.86,42.701038


Fourth indicator: MACD, the gap between a fast (12-day) and slow (26-day) EMA, along with its 9-day signal line. The table below shows all the new indicator columns for the last few AAPL days, so we can eyeball that the values look reasonable.

In [25]:
prices = (
    prices
    .with_columns(
        (pl.col("close").ewm_mean(span=12, adjust=False).over("name")
         - pl.col("close").ewm_mean(span=26, adjust=False).over("name")).alias("macd")
    )
    .with_columns(pl.col("macd").ewm_mean(span=9, adjust=False).over("name").alias("macd_signal"))
)
(prices.filter(pl.col("name") == "AAPL")
       .select(["date", "close", "sma_20", "ema_20", "rsi_14", "macd", "macd_signal"])
       .tail(5))

date,close,sma_20,ema_20,rsi_14,macd,macd_signal
date,f64,f64,f64,f64,f64,f64
2018-02-01,167.78,173.87,172.263816,35.765808,-1.249257,-0.035996
2018-02-02,160.5,173.2435,171.143452,24.836304,-2.012771,-0.431351
2018-02-05,156.49,172.318,169.747885,21.025056,-2.907914,-0.926663
2018-02-06,163.03,171.752,169.108087,37.791716,-3.054389,-1.352208
2018-02-07,159.54,171.0125,168.19684,33.682205,-3.412745,-1.764316


Now we set up the prediction problem. Instead of predicting the raw price, which differs hugely between a $20 stock and a $1,000 one, the models predict the next-day percentage change and we convert it back to a price. The inputs are scale-free versions of the indicators, so one model can serve all 505 companies. The very last row of each company has no "tomorrow" yet, so we set those aside as the true forecast rows.

In [28]:
prices = prices.with_columns(
    (pl.col("close") / pl.col("sma_20") - 1).alias("close_vs_sma"),
    (pl.col("close") / pl.col("ema_20") - 1).alias("close_vs_ema"),
    (pl.col("macd") / pl.col("close")).alias("macd_pct"),
    ((pl.col("macd") - pl.col("macd_signal")) / pl.col("close")).alias("macd_hist_pct"),
    pl.col("close").shift(-1).over("name").alias("next_close"),
    pl.col("date").shift(-1).over("name").alias("pred_date"),
).with_columns(
    (pl.col("next_close") / pl.col("close") - 1).alias("target_ret")
)

FEATURES = ["ret_1", "ret_5", "ret_10", "close_vs_sma", "close_vs_ema",
            "rsi_14", "macd_pct", "macd_hist_pct", "vol_ratio"]

# Drop the warm-up rows where rolling windows are not full yet
model_df = (
    prices.drop_nulls(FEATURES)
          .filter(pl.all_horizontal(pl.col(FEATURES).is_finite()))
)

labelled = model_df.filter(pl.col("next_close").is_not_null())
latest   = model_df.filter(pl.col("next_close").is_null())
print(f"Rows with a known next-day price: {labelled.height:,}")
print(f"Rows reserved for the real forecast: {latest.height}")

Rows with a known next-day price: 608,940
Rows reserved for the real forecast: 505


The 80/20 split is done by date rather than at random, so the models are always tested on days that come after everything they were trained on. That is how back-testing should work for time series. The table confirms the date ranges of both sets.

In [31]:
dates  = labelled["date"].unique().sort()
cutoff = dates[int(len(dates) * 0.8)]

train = labelled.filter(pl.col("date") < cutoff)
test  = labelled.filter(pl.col("date") >= cutoff)

pl.DataFrame({
    "set":  ["train", "test"],
    "rows": [train.height, test.height],
    "from": [train["date"].min(), test["date"].min()],
    "to":   [train["date"].max(), test["date"].max()],
})

set,rows,from,to
str,i64,date,date
"""train""",484377,2013-03-08,2017-02-10
"""test""",124563,2017-02-13,2018-02-06


Two models are trained on the same features. The first is a Ridge regression, a simple, regularised linear model that gives a solid, easy-to-explain starting point. The second is gradient boosting, which can capture non-linear patterns the linear model misses.

In [36]:
X_train, y_train = train.select(FEATURES).to_numpy(), train["target_ret"].to_numpy()
X_test           = test.select(FEATURES).to_numpy()

ridge = make_pipeline(StandardScaler(), Ridge(alpha=1.0)).fit(X_train, y_train)

hgb = HistGradientBoostingRegressor(
    max_iter=300, learning_rate=0.05, max_depth=6,
    l2_regularization=1.0, random_state=42,
).fit(X_train, y_train)

print("Both models trained.")

Both models trained.


Time to back-test. Each model's predicted return is turned into a predicted price and compared with the real next-day close on the test period. We also include a "naive" baseline that just says tomorrow's price equals today's, which gives us something honest to beat.

In [39]:
def score(y_true, y_pred):
    err = y_pred - y_true
    return {
        "RMSE":   float(np.sqrt(np.mean(err ** 2))),
        "MAE":    float(np.mean(np.abs(err))),
        "MAPE_%": float(np.mean(np.abs(err / y_true)) * 100),
    }

y_true    = test["next_close"].to_numpy()
close_now = test["close"].to_numpy()

test_preds = {
    "Naive (today's close)": close_now,
    "Ridge regression":      close_now * (1 + ridge.predict(X_test)),
    "Gradient boosting":     close_now * (1 + hgb.predict(X_test)),
}

results = pl.DataFrame([{"model": name, **score(y_true, p)} for name, p in test_preds.items()])
results.with_columns(pl.col(pl.Float64).round(4))

model,RMSE,MAE,MAPE_%
str,f64,f64,f64
"""Naive (today's close)""",2.2281,0.9052,0.9446
"""Ridge regression""",2.2283,0.9047,0.9446
"""Gradient boosting""",2.2257,0.9051,0.9452


Using the models trained above, we now produce the actual next-day forecast for every company from its latest row, and package the back-test predictions too. Everything the dashboard needs is saved to small Parquet files so the dashboard never has to re-run the models.

In [42]:
# Back-test predictions (one row per company per test day)
test_out = test.select(["name", "date", "pred_date", "close", "next_close"]).with_columns(
    pl.Series("pred_ridge", test_preds["Ridge regression"]),
    pl.Series("pred_hgb",   test_preds["Gradient boosting"]),
)

# True forecast for the first trading day after the data ends
X_latest  = latest.select(FEATURES).to_numpy()
last_date = latest["date"].max()
next_day  = np.busday_offset(np.datetime64(last_date), 1).astype(object)

forecast = latest.select(["name", "date", "close"]).with_columns(
    pl.lit(next_day).alias("forecast_date"),
    pl.Series("forecast_ridge", latest["close"].to_numpy() * (1 + ridge.predict(X_latest))),
    pl.Series("forecast_hgb",   latest["close"].to_numpy() * (1 + hgb.predict(X_latest))),
)

model_df.select(["name", "date", "close", "sma_20", "ema_20", "rsi_14", "macd", "macd_signal"]) \
        .write_parquet(FEATURE_FILE, compression="zstd")
test_out.write_parquet(PRED_FILE, compression="zstd")
forecast.write_parquet(FORECAST_FILE, compression="zstd")

forecast.head()

name,date,close,forecast_date,forecast_ridge,forecast_hgb
str,date,f64,date,f64,f64
"""A""",2018-02-07,68.06,2018-02-08,68.152032,68.233933
"""AAL""",2018-02-07,51.4,2018-02-08,51.509798,51.524465
"""AAP""",2018-02-07,109.93,2018-02-08,110.005725,111.246584
"""AAPL""",2018-02-07,159.54,2018-02-08,159.811687,159.895251
"""ABBV""",2018-02-07,113.62,2018-02-08,113.604093,113.621959


## Part 3 – Dashboard

The dashboard reads the three small Parquet files from Part 2. Pick or type a ticker in the dropdown and the charts update: back-tested predictions against the actual price (with tomorrow's forecast marked), the price with its moving averages, and the RSI. Each update only pulls the selected company's rows from disk, so it stays quick as the data grows.

In [45]:
from dash import Dash, dcc, html, Input, Output

tickers = (pl.scan_parquet(FORECAST_FILE).select("name").unique().sort("name")
             .collect()["name"].to_list())

def rmse(a, b):
    return float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))

app = Dash(__name__)
app.layout = html.Div(
    style={"fontFamily": "Arial, sans-serif", "maxWidth": "1100px", "margin": "auto"},
    children=[
        html.H2("S&P 500 – Next-Day Closing Price Predictions"),
        html.Label("Search or select a company ticker"),
        dcc.Dropdown(id="ticker", options=tickers,
                     value="AAPL" if "AAPL" in tickers else tickers[0], clearable=False),
        html.Div(id="summary", style={"margin": "14px 0", "fontSize": "15px"}),
        dcc.Graph(id="pred-chart"),
        dcc.Graph(id="trend-chart"),
        dcc.Graph(id="rsi-chart"),
    ],
)

@app.callback(
    Output("summary", "children"),
    Output("pred-chart", "figure"),
    Output("trend-chart", "figure"),
    Output("rsi-chart", "figure"),
    Input("ticker", "value"),
)
def update(ticker):
    # Only this company's rows are read from disk
    hist = pl.scan_parquet(FEATURE_FILE).filter(pl.col("name") == ticker).collect()
    pred = pl.scan_parquet(PRED_FILE).filter(pl.col("name") == ticker).collect()
    fc   = pl.scan_parquet(FORECAST_FILE).filter(pl.col("name") == ticker).collect().row(0, named=True)

    summary = (f"Last close ({fc['date']}): {fc['close']:.2f}  |  "
               f"Forecast for {fc['forecast_date']} – Ridge: {fc['forecast_ridge']:.2f}, "
               f"Gradient boosting: {fc['forecast_hgb']:.2f}  |  "
               f"Test RMSE – Ridge: {rmse(pred['next_close'], pred['pred_ridge']):.2f}, "
               f"Gradient boosting: {rmse(pred['next_close'], pred['pred_hgb']):.2f}")

    f1 = go.Figure()
    f1.add_scatter(x=pred["pred_date"], y=pred["next_close"], name="Actual close")
    f1.add_scatter(x=pred["pred_date"], y=pred["pred_ridge"], name="Ridge prediction")
    f1.add_scatter(x=pred["pred_date"], y=pred["pred_hgb"], name="Gradient boosting prediction")
    f1.add_scatter(x=[fc["forecast_date"]] * 2, y=[fc["forecast_ridge"], fc["forecast_hgb"]],
                   mode="markers", marker={"size": 11, "symbol": "star"}, name="Next-day forecast")
    f1.update_layout(title=f"{ticker}: actual vs predicted close (test period)",
                     yaxis_title="Price", hovermode="x unified")

    f2 = go.Figure()
    f2.add_scatter(x=hist["date"], y=hist["close"], name="Close")
    f2.add_scatter(x=hist["date"], y=hist["sma_20"], name="SMA 20")
    f2.add_scatter(x=hist["date"], y=hist["ema_20"], name="EMA 20")
    f2.update_layout(title=f"{ticker}: price with moving averages", yaxis_title="Price",
                     hovermode="x unified")

    f3 = go.Figure()
    f3.add_scatter(x=hist["date"], y=hist["rsi_14"], name="RSI 14")
    f3.add_hline(y=70, line_dash="dash")
    f3.add_hline(y=30, line_dash="dash")
    f3.update_layout(title=f"{ticker}: RSI (14)", yaxis_title="RSI", yaxis_range=[0, 100])

    return summary, f1, f2, f3

app.run(jupyter_mode="inline", jupyter_height=1350, debug=False)

That wraps up the prototype: data stored in Parquet, indicators and two models built with Polars, and a searchable dashboard covering every company.